# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullahhashmi01/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
from sklearn.datasets import load_breast_cancer

csv_path = "starter_breast_cancer.csv"

# Make the notebook independently runnable
if not os.path.exists(csv_path):
    dataset = load_breast_cancer(as_frame=True)
    data = dataset.frame.copy()

    # Original dataset mapping:
    # 0 = malignant and 1 = benign
    data["diagnosis"] = data["target"].map({
        0: "malignant",
        1: "benign"
    })

    data.to_csv(csv_path, index=False)

# Load the CSV
df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (569, 32)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,malignant


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a clear machine-learning target
df["malignant_flag"] = (
    df["diagnosis"] == "malignant"
).astype(int)

target_column = "malignant_flag"

print("Target encoding:")
print("1 = malignant")
print("0 = benign")

print("\nTarget counts:")
print(df[target_column].value_counts().sort_index())

print("\nTarget percentages:")
print(
    df[target_column]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(1)
)

Target encoding:
1 = malignant
0 = benign

Target counts:
malignant_flag
0    357
1    212
Name: count, dtype: int64

Target percentages:
malignant_flag
0    62.7
1    37.3
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import fbeta_score

def malignant_f2_score(y_true, y_pred):
    """
    Calculate the F2-score for the malignant class.

    malignant = 1
    benign = 0
    """
    return fbeta_score(
        y_true,
        y_pred,
        beta=2,
        pos_label=1
    )

success_threshold = 0.90

print(f"Primary metric: F2-score")
print(f"Provisional success threshold: {success_threshold:.2f}")


Primary metric: F2-score
Provisional success threshold: 0.90


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Select the feature columns used in this lane
feature_columns = [
    column for column in df.columns
    if column not in [
        "target",
        "diagnosis",
        "malignant_flag"
    ]
]

# X contains model inputs
X = df[feature_columns].copy()

# y contains the prediction target
y = df[target_column].copy()

# Real dataframe slice for inspection
lane_slice = df[
    [
        "mean radius",
        "mean texture",
        "mean perimeter",
        "mean area",
        "mean concave points",
        "malignant_flag"
    ]
].copy()

print("One row = one image-based breast-mass observation")
print("Number of observations:", X.shape[0])
print("Number of model features:", X.shape[1])
print("Target column:", target_column)

display(lane_slice.head())

One row = one image-based breast-mass observation
Number of observations: 569
Number of model features: 30
Target column: malignant_flag


,mean radius,mean texture,mean perimeter,mean area,mean concave points,malignant_flag
0,17.99,10.38,122.80,1001.0,0.14710,1
1,20.57,17.77,132.90,1326.0,0.07017,1
2,19.69,21.25,130.00,1203.0,0.12790,1
3,11.42,20.38,77.58,386.1,0.10520,1
4,20.29,14.34,135.10,1297.0,0.10430,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic checks
assert X.shape == (569, 30)
assert len(y) == 569
assert y.isin([0, 1]).all()
assert list(X.index) == list(y.index)

print("The feature and target dataframes are aligned.")
print("All target values are valid.")


The feature and target dataframes are aligned.
All target values are valid.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.